# Advanced Problems with Solutions: Slicing Iterables in Python

**Topic:** slicing ordinary sequences, slicing lazy iterables with `itertools.islice`, iterator consumption, infinite streams, and reusable slicing patterns.

This notebook is designed as a problem set. Each problem includes:

1. a clear task,
2. constraints and best-practice notes,
3. a full solution,
4. runnable checks using `assert`.

> Recommended use: run cells top-to-bottom once, then re-run individual problem cells while experimenting.


## Learning goals

By the end, you should be able to:

- explain why lists can be sliced but generators cannot be indexed directly,
- use `slice` objects and `itertools.islice`,
- safely slice finite and infinite iterables,
- predict how many items an iterator has consumed,
- build reusable helpers such as `take`, `drop`, `nth`, `chunked`, `windowed`, and `peek`,
- write tests that verify lazy iterator behavior.


## Setup


In [1]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from itertools import chain, count, islice, tee
from typing import Callable, Iterable, Iterator, TypeVar, Optional, Any
import math

T = TypeVar("T")
U = TypeVar("U")


In [2]:
def factorials(limit: Optional[int] = None, *, trace: bool = False) -> Iterator[int]:
    """Yield factorial numbers.

    If limit is None, this is an infinite iterator:
        0!, 1!, 2!, 3!, ...

    If limit is an integer, yields factorials for range(limit).
    """
    i = 0
    while limit is None or i < limit:
        if trace:
            print(f"yielding factorial({i})")
        yield math.factorial(i)
        i += 1


def squares() -> Iterator[int]:
    """Infinite stream of square numbers: 0, 1, 4, 9, ..."""
    for n in count(0):
        yield n * n


def first_values(iterable: Iterable[T], n: int) -> list[T]:
    """Small display helper used throughout the notebook."""
    return list(islice(iterable, n))


## Warm-up: sequence slicing vs iterable slicing

A **sequence** such as a list supports indexing and slicing because it implements `__getitem__`.

A **generator** is an iterator, not a sequence. It yields values one at a time and does not support direct slicing.


In [3]:
values = [10, 20, 30, 40, 50, 60]

assert values[1:4] == [20, 30, 40]
assert values[slice(1, 4)] == [20, 30, 40]
assert values[::2] == [10, 30, 50]

values[1:4], values[slice(1, 4)], values[::2]


([20, 30, 40], [20, 30, 40], [10, 30, 50])

In [4]:
g = factorials(10)

try:
    g[0:3]
except TypeError as ex:
    error_message = str(ex)

assert "not subscriptable" in error_message
error_message


"'generator' object is not subscriptable"

`itertools.islice` solves this by consuming an iterable lazily and yielding the requested items.

Important differences from normal sequence slicing:

- negative indices are not supported,
- negative or zero step values are not supported,
- the returned object is itself an iterator,
- consuming an `islice` also consumes the underlying iterator.


In [5]:
assert list(islice(factorials(10), 0, 5)) == [1, 1, 2, 6, 24]
assert list(islice(factorials(10), 0, 10, 2)) == [1, 2, 24, 720, 40320]
assert list(islice(factorials(), 5)) == [1, 1, 2, 6, 24]


## Utility: tracing iterator consumption

Several advanced problems require checking not only *what* is yielded, but also *how much* of the source iterator was consumed.


In [6]:
@dataclass
class TracedIterator(Iterator[T]):
    """Wrap an iterable and count how many times __next__ is called successfully."""
    iterable: Iterable[T]

    def __post_init__(self) -> None:
        self._iterator = iter(self.iterable)
        self.next_calls = 0
        self.values_returned: list[T] = []

    def __iter__(self) -> "TracedIterator[T]":
        return self

    def __next__(self) -> T:
        value = next(self._iterator)
        self.next_calls += 1
        self.values_returned.append(value)
        return value


In [7]:
traced = TracedIterator(range(10))
result = list(islice(traced, 2, 8, 2))

assert result == [2, 4, 6]
assert traced.next_calls == 8
assert traced.values_returned == list(range(8))

result, traced.next_calls, traced.values_returned


([2, 4, 6], 8, [0, 1, 2, 3, 4, 5, 6, 7])

---

# Problem 1 — Implement `take`

## Task

Write a function `take(iterable, n)` that returns the first `n` items from **any** iterable as a list.

## Constraints

- It must work with lists, tuples, generators, and infinite iterators.
- It must not materialize more than `n` items.
- For `n <= 0`, return an empty list.


## Solution 1


In [8]:
def take(iterable: Iterable[T], n: int) -> list[T]:
    """Return the first n values from iterable as a list."""
    if n <= 0:
        return []
    return list(islice(iterable, n))


assert take([1, 2, 3, 4], 2) == [1, 2]
assert take((x * 10 for x in range(5)), 3) == [0, 10, 20]
assert take(count(100), 4) == [100, 101, 102, 103]
assert take(count(), 0) == []
assert take(count(), -5) == []


---

# Problem 2 — Implement `drop`

## Task

Write `drop(iterable, n)` that skips the first `n` items and returns a lazy iterator over the rest.

## Constraints

- Do not convert the whole iterable into a list.
- It must support infinite iterables.
- For `n <= 0`, return the original stream from the beginning.


## Solution 2


In [9]:
def drop(iterable: Iterable[T], n: int) -> Iterator[T]:
    """Lazily skip the first n values."""
    if n <= 0:
        return iter(iterable)
    return islice(iterable, n, None)


assert take(drop([10, 20, 30, 40], 2), 10) == [30, 40]
assert take(drop(count(10), 5), 4) == [15, 16, 17, 18]
assert take(drop(range(3), 99), 5) == []
assert take(drop(range(3), -1), 5) == [0, 1, 2]


---

# Problem 3 — Implement `every_nth`

## Task

Write `every_nth(iterable, n, start=0)` that yields every `n`-th item from an iterable, beginning at index `start`.

For example:

```python
every_nth(range(10), 3, start=1) -> 1, 4, 7
```

## Constraints

- `n` must be positive.
- The function must be lazy.
- It must work with infinite streams.


## Solution 3


In [10]:
def every_nth(iterable: Iterable[T], n: int, *, start: int = 0) -> Iterator[T]:
    """Yield every n-th item starting at zero-based position start."""
    if n <= 0:
        raise ValueError("n must be positive")
    if start < 0:
        raise ValueError("start must be non-negative")
    return islice(iterable, start, None, n)


assert list(every_nth(range(10), 3, start=1)) == [1, 4, 7]
assert take(every_nth(count(0), 100, start=25), 5) == [25, 125, 225, 325, 425]

try:
    list(every_nth(range(10), 0))
except ValueError as ex:
    assert "positive" in str(ex)


---

# Problem 4 — Recreate a safe subset of `islice`

## Task

Implement `my_islice(iterable, *args)` supporting the same call shapes as `itertools.islice`:

```python
my_islice(iterable, stop)
my_islice(iterable, start, stop)
my_islice(iterable, start, stop, step)
```

## Constraints

- `start`, `stop`, and `step` may be `None` where `islice` allows it.
- Negative `start` or `stop` must raise `ValueError`.
- `step <= 0` must raise `ValueError`.
- The function must be lazy.
- Be careful not to consume one item too many at `stop`.


## Solution 4


In [11]:
def my_islice(iterable: Iterable[T], *args: Optional[int]) -> Iterator[T]:
    """A learning implementation of a safe subset of itertools.islice."""
    if not 1 <= len(args) <= 3:
        raise TypeError("my_islice expected 2 to 4 arguments including iterable")

    # Convert call shapes into start, stop, step.
    if len(args) == 1:
        start, stop, step = 0, args[0], 1
    elif len(args) == 2:
        start, stop = args
        step = 1
    else:
        start, stop, step = args

    start = 0 if start is None else start
    step = 1 if step is None else step

    if start < 0:
        raise ValueError("start must be non-negative")
    if stop is not None and stop < 0:
        raise ValueError("stop must be non-negative or None")
    if step <= 0:
        raise ValueError("step must be positive")

    iterator = iter(iterable)

    # Consume exactly start items, unless the source is exhausted first.
    for _ in range(start):
        try:
            next(iterator)
        except StopIteration:
            return

    # Now index is the absolute index of the next item to read.
    index = start
    next_yield_index = start

    while stop is None or index < stop:
        try:
            value = next(iterator)
        except StopIteration:
            return

        if index == next_yield_index:
            yield value
            next_yield_index += step

        index += 1


cases = [
    (range(20), (5,)),
    (range(20), (2, 10)),
    (range(20), (2, 10, 3)),
    (range(20), (None, 10, 2)),
    (range(20), (5, None, 4)),
]

for iterable, args in cases:
    assert list(my_islice(iterable, *args)) == list(islice(iterable, *args))

assert take(my_islice(count(), 10, None, 5), 4) == [10, 15, 20, 25]

for bad_args in [(-1, 10), (1, -10), (1, 10, 0), (1, 10, -2)]:
    try:
        list(my_islice(range(20), *bad_args))
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected ValueError for args={bad_args}")


---

# Problem 5 — Prove how many values were consumed

## Task

Given this expression:

```python
list(islice(source, 3, 12, 4))
```

show which values are yielded and how many values are consumed from `source`.

## Best-practice note

When slicing an iterator, skipped values are still consumed. `islice(source, 3, 12, 4)` must advance through positions `0` to `11`.


## Solution 5


In [12]:
source = TracedIterator(range(100))
yielded = list(islice(source, 3, 12, 4))

assert yielded == [3, 7, 11]
assert source.next_calls == 12
assert source.values_returned == list(range(12))

yielded, source.next_calls, source.values_returned


([3, 7, 11], 12, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])

---

# Problem 6 — Page a log stream

## Task

A log stream yields one log line at a time. Implement:

```python
get_page(lines, page_number, page_size)
```

where `page_number` is zero-based.

For example, page `2` of size `5` should return indices `10..14`.

## Constraints

- Must work with generators.
- Must consume only up to the end of the requested page.
- Validate inputs.


## Solution 6


In [13]:
def fake_log_stream(total: Optional[int] = None) -> Iterator[str]:
    i = 0
    while total is None or i < total:
        yield f"line-{i:04d}: event processed"
        i += 1


def get_page(lines: Iterable[str], page_number: int, page_size: int) -> list[str]:
    if page_number < 0:
        raise ValueError("page_number must be non-negative")
    if page_size <= 0:
        raise ValueError("page_size must be positive")

    start = page_number * page_size
    stop = start + page_size
    return list(islice(lines, start, stop))


page = get_page(fake_log_stream(), page_number=2, page_size=5)

assert page == [
    "line-0010: event processed",
    "line-0011: event processed",
    "line-0012: event processed",
    "line-0013: event processed",
    "line-0014: event processed",
]

assert get_page(fake_log_stream(3), page_number=1, page_size=5) == []
page


['line-0010: event processed',
 'line-0011: event processed',
 'line-0012: event processed',
 'line-0013: event processed',
 'line-0014: event processed']

---

# Problem 7 — Batch an iterable lazily

## Task

Implement `batched(iterable, size)` that yields tuples of length `size`, except possibly the final tuple.

Example:

```python
list(batched(range(10), 3))
# [(0, 1, 2), (3, 4, 5), (6, 7, 8), (9,)]
```

## Constraints

- Use `islice`.
- Must be lazy.
- Must support generators.
- `size` must be positive.


## Solution 7


In [14]:
def batched(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    while True:
        batch = tuple(islice(iterator, size))
        if not batch:
            return
        yield batch


assert list(batched(range(10), 3)) == [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    (9,),
]

assert list(batched((x * x for x in range(7)), 2)) == [(0, 1), (4, 9), (16, 25), (36,)]
assert take(batched(count(1), 4), 3) == [(1, 2, 3, 4), (5, 6, 7, 8), (9, 10, 11, 12)]


---

# Problem 8 — Preview an iterator without losing data

## Task

Write `peek(iterable, n)` that returns:

```python
preview_values, replayable_iterable
```

The preview should contain the first `n` items, but the returned iterable should still produce those same first items when consumed.

## Best-practice note

`itertools.tee` can split an iterable into independent iterators. It is useful, but it can buffer data internally if one copy advances far ahead of the other.


## Solution 8


In [15]:
def peek(iterable: Iterable[T], n: int) -> tuple[list[T], Iterator[T]]:
    if n < 0:
        raise ValueError("n must be non-negative")

    preview_iter, replay_iter = tee(iterable, 2)
    preview = list(islice(preview_iter, n))
    return preview, replay_iter


source = (x * 10 for x in range(5))
preview, replay = peek(source, 3)

assert preview == [0, 10, 20]
assert list(replay) == [0, 10, 20, 30, 40]

preview


[0, 10, 20]

---

# Problem 9 — Preserve consumed preview manually

## Task

Solve the preview problem again, but this time do **not** use `tee`.

Write `peek_chain(iterable, n)` that consumes at most `n` items, stores them in a small list, and returns:

```python
preview_values, new_iterator
```

where `new_iterator` yields the previewed values first, then continues the original iterator.


## Solution 9


In [16]:
def peek_chain(iterable: Iterable[T], n: int) -> tuple[list[T], Iterator[T]]:
    if n < 0:
        raise ValueError("n must be non-negative")

    iterator = iter(iterable)
    preview = list(islice(iterator, n))
    return preview, chain(preview, iterator)


source = (x + 1 for x in range(5))
preview, replay = peek_chain(source, 2)

assert preview == [1, 2]
assert list(replay) == [1, 2, 3, 4, 5]

preview


[1, 2]

---

# Problem 10 — Extract the last `n` values from a finite iterable

## Task

Normal list slicing supports negative indices:

```python
values[-3:]
```

`islice` does not. Write `tail(iterable, n)` for finite iterables.

## Constraints

- Must work with iterators and generators.
- Must not assume the iterable has `len`.
- Must keep only `n` values in memory.
- For `n <= 0`, return an empty list.


## Solution 10


In [17]:
def tail(iterable: Iterable[T], n: int) -> list[T]:
    if n <= 0:
        return []
    return list(deque(iterable, maxlen=n))


assert tail([1, 2, 3, 4, 5], 3) == [3, 4, 5]
assert tail((x * x for x in range(6)), 2) == [16, 25]
assert tail(range(3), 10) == [0, 1, 2]
assert tail(range(3), 0) == []


---

# Problem 11 — Last `n` values before a stop index

## Task

Write `tail_before(iterable, stop, n)` that returns the last `n` values among the first `stop` values of `iterable`.

Example:

```python
tail_before(range(20), stop=10, n=3) -> [7, 8, 9]
```

## Constraint

Use `islice` to avoid consuming beyond `stop`.


## Solution 11


In [18]:
def tail_before(iterable: Iterable[T], stop: int, n: int) -> list[T]:
    if stop < 0:
        raise ValueError("stop must be non-negative")
    if n <= 0:
        return []
    return tail(islice(iterable, stop), n)


traced = TracedIterator(range(100))
result = tail_before(traced, stop=10, n=3)

assert result == [7, 8, 9]
assert traced.next_calls == 10

result, traced.next_calls


([7, 8, 9], 10)

---

# Problem 12 — Build sliding windows lazily

## Task

Write `windowed(iterable, size)` that yields overlapping windows.

Example:

```python
list(windowed([1, 2, 3, 4], 3))
# [(1, 2, 3), (2, 3, 4)]
```

## Constraints

- Use `islice` for the initial window.
- Must work with generators.
- Must be lazy after the initial window.


## Solution 12


In [19]:
def windowed(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    window = deque(islice(iterator, size), maxlen=size)

    if len(window) < size:
        return

    yield tuple(window)

    for value in iterator:
        window.append(value)
        yield tuple(window)


assert list(windowed([1, 2, 3, 4], 3)) == [(1, 2, 3), (2, 3, 4)]
assert list(windowed(range(2), 3)) == []
assert take(windowed(count(1), 4), 3) == [
    (1, 2, 3, 4),
    (2, 3, 4, 5),
    (3, 4, 5, 6),
]


---

# Problem 13 — Read selected columns from row iterables

## Task

A stream yields rows as tuples. Write `select_columns(rows, column_slice)` that slices each row using a normal `slice` object.

Then combine it with `islice` to read only a subset of rows.

## Constraints

- Rows are sequences, so normal slicing is allowed inside each row.
- The row stream itself may be a generator, so use `islice` for row selection.


## Solution 13


In [20]:
def row_stream() -> Iterator[tuple[int, int, int, int, int]]:
    for i in count(0):
        yield (i, i + 10, i + 20, i + 30, i + 40)


def select_columns(rows: Iterable[tuple[Any, ...]], column_slice: slice) -> Iterator[tuple[Any, ...]]:
    for row in rows:
        yield row[column_slice]


selected_rows = islice(row_stream(), 3, 8)
selected_columns = list(select_columns(selected_rows, slice(1, 4)))

assert selected_columns == [
    (13, 23, 33),
    (14, 24, 34),
    (15, 25, 35),
    (16, 26, 36),
    (17, 27, 37),
]

selected_columns


[(13, 23, 33), (14, 24, 34), (15, 25, 35), (16, 26, 36), (17, 27, 37)]

---

# Problem 14 — Find records after a marker

## Task

A stream contains events. Write `after_first_match(iterable, predicate)` that skips through the stream until the first matching item is found, then yields everything after that matching item.

Then use `islice` to return only the next `k` records.

## Constraints

- The matching record itself should not be yielded.
- The function should stop searching as soon as it finds a match.
- It must work with infinite streams.


## Solution 14


In [21]:
def after_first_match(iterable: Iterable[T], predicate: Callable[[T], bool]) -> Iterator[T]:
    iterator = iter(iterable)

    for item in iterator:
        if predicate(item):
            break
    else:
        return

    yield from iterator


events = (f"event-{i}" for i in count())
after_event_10 = after_first_match(events, lambda s: s == "event-10")
next_five = list(islice(after_event_10, 5))

assert next_five == ["event-11", "event-12", "event-13", "event-14", "event-15"]
next_five


['event-11', 'event-12', 'event-13', 'event-14', 'event-15']

---

# Problem 15 — Implement `nth`

## Task

Write `nth(iterable, index, default=None)` that returns the item at zero-based `index`.

## Constraints

- Must work with generators and infinite iterators.
- Must not materialize the iterable.
- If the item does not exist, return `default`.


## Solution 15


In [22]:
def nth(iterable: Iterable[T], index: int, default: Optional[T] = None) -> Optional[T]:
    if index < 0:
        raise ValueError("index must be non-negative")
    return next(islice(iterable, index, None), default)


assert nth(range(10), 0) == 0
assert nth(range(10), 5) == 5
assert nth((x * 3 for x in range(10)), 4) == 12
assert nth(range(3), 99, default="missing") == "missing"
assert nth(count(10), 1_000_000) == 1_000_010


---

# Problem 16 — Avoid accidental double consumption

## Task

The following pattern is a common bug:

```python
s = islice(iterator, 3)
list(s)
list(s)
```

The second list is empty because `s` is an iterator and has already been exhausted.

Write a short demonstration and then fix it by creating a fresh `islice` from a re-iterable source.


## Solution 16


In [23]:
# Bug demonstration: islice object is single-use.
source = [1, 2, 3, 4, 5]
s = islice(source, 3)

first_pass = list(s)
second_pass = list(s)

assert first_pass == [1, 2, 3]
assert second_pass == []

# Fix: create a new islice each time when the source is re-iterable.
def first_three_again() -> list[int]:
    return list(islice(source, 3))

assert first_three_again() == [1, 2, 3]
assert first_three_again() == [1, 2, 3]

first_pass, second_pass, first_three_again()


([1, 2, 3], [], [1, 2, 3])

---

# Problem 17 — Build a replayable lazy slice factory

## Task

Sometimes you need the same slice more than once. A plain iterator cannot be replayed, but a **factory** that creates a new iterator can.

Write `slice_factory(iterable_factory, *islice_args)` that returns a function. Each call to that returned function should produce a fresh list of sliced values.

## Constraints

- The input must be a callable that returns a new iterable each time.
- Use `islice` internally.


## Solution 17


In [24]:
def slice_factory(
    iterable_factory: Callable[[], Iterable[T]],
    *islice_args: Optional[int],
) -> Callable[[], list[T]]:
    def run_slice() -> list[T]:
        return list(islice(iterable_factory(), *islice_args))
    return run_slice


fresh_factorials = lambda: factorials(10)
get_middle_factorials = slice_factory(fresh_factorials, 3, 7)

assert get_middle_factorials() == [6, 24, 120, 720]
assert get_middle_factorials() == [6, 24, 120, 720]

get_middle_factorials()


[6, 24, 120, 720]

---

# Problem 18 — Create a lazy processing pipeline

## Task

Build a pipeline that:

1. starts with natural numbers `0, 1, 2, ...`,
2. squares them,
3. skips the first `10` squared values,
4. takes every `5`th value,
5. returns the first `8` results.

## Constraint

The pipeline should stay lazy until the final `list(...)` call.


## Solution 18


In [25]:
def lazy_square_pipeline() -> list[int]:
    numbers = count(0)
    squared = (n * n for n in numbers)
    sampled = islice(squared, 10, None, 5)
    return list(islice(sampled, 8))


result = lazy_square_pipeline()
expected = [10**2, 15**2, 20**2, 25**2, 30**2, 35**2, 40**2, 45**2]

assert result == expected
result


[100, 225, 400, 625, 900, 1225, 1600, 2025]

---

# Problem 19 — Sample from a large file-like stream

## Task

Simulate a file as an iterator of lines. Write `sample_lines(lines, start, stop, step)` that returns the selected lines without loading the whole file.

## Constraints

- Use `islice`.
- Strip trailing newlines.
- Validate that `step` is positive.


## Solution 19


In [26]:
def simulated_file(num_lines: int) -> Iterator[str]:
    for i in range(num_lines):
        yield f"record_id={i},status=ok\n"


def sample_lines(lines: Iterable[str], start: int, stop: Optional[int], step: int) -> list[str]:
    if start < 0:
        raise ValueError("start must be non-negative")
    if stop is not None and stop < 0:
        raise ValueError("stop must be non-negative or None")
    if step <= 0:
        raise ValueError("step must be positive")

    return [line.rstrip("\n") for line in islice(lines, start, stop, step)]


sample = sample_lines(simulated_file(100), 10, 25, 5)

assert sample == [
    "record_id=10,status=ok",
    "record_id=15,status=ok",
    "record_id=20,status=ok",
]

sample


['record_id=10,status=ok', 'record_id=15,status=ok', 'record_id=20,status=ok']

---

# Problem 20 — Compare list slicing and iterator slicing side effects

## Task

Show the behavioral difference between slicing a list and slicing an iterator:

- list slicing does not mutate or consume the list,
- `islice` consumes the underlying iterator.


## Solution 20


In [27]:
# List slicing: no consumption.
values = [0, 1, 2, 3, 4, 5]
first = values[0:3]
second = values[0:3]

assert first == [0, 1, 2]
assert second == [0, 1, 2]
assert values == [0, 1, 2, 3, 4, 5]

# Iterator slicing: consumption happens.
iterator = iter(values)
first_iter_slice = list(islice(iterator, 0, 3))
second_iter_slice = list(islice(iterator, 0, 3))

assert first_iter_slice == [0, 1, 2]
assert second_iter_slice == [3, 4, 5]

first, second, first_iter_slice, second_iter_slice


([0, 1, 2], [0, 1, 2], [0, 1, 2], [3, 4, 5])

---

# Problem 21 — Validate custom `my_islice` with randomized-style cases

## Task

Create several deterministic test cases comparing `my_islice` to `itertools.islice`.

## Best-practice note

When reusing an iterable in tests, use a fresh source each time. Do **not** compare two slices over the same iterator, because the first test will consume it.


## Solution 21


In [28]:
test_args = [
    (0,),
    (1,),
    (5,),
    (0, 5),
    (2, 8),
    (2, 8, 2),
    (None, 8, 3),
    (4, None, 5),
    (None, None, 7),
]

for args in test_args:
    expected = list(islice(range(30), *args))
    actual = list(my_islice(range(30), *args))
    assert actual == expected, (args, actual, expected)

# Also test with a generator source.
for args in test_args:
    expected = list(islice((x * 2 for x in range(30)), *args))
    actual = list(my_islice((x * 2 for x in range(30)), *args))
    assert actual == expected, (args, actual, expected)

"All my_islice tests passed"


'All my_islice tests passed'

---

# Problem 22 — Design a memory-safe `take_until_size` helper

## Task

Suppose an infinite stream yields strings of variable length. Write `take_until_size(iterable, max_total_chars)` that yields items until adding the next item would exceed the character budget.

## Constraints

- Must be lazy.
- Must stop as soon as the budget would be exceeded.
- Must not consume the item that would exceed the budget if you need to preserve it. This is impossible with a plain iterator unless you add buffering, so document the behavior clearly.

## Best-practice note

A plain iterator cannot be "un-nexted." Once you inspect an item, it is consumed. This solution consumes the first item that breaks the budget.


## Solution 22


In [29]:
def take_until_size(iterable: Iterable[str], max_total_chars: int) -> Iterator[str]:
    if max_total_chars < 0:
        raise ValueError("max_total_chars must be non-negative")

    total = 0
    for item in iterable:
        proposed_total = total + len(item)
        if proposed_total > max_total_chars:
            return
        yield item
        total = proposed_total


words = iter(["alpha", "beta", "gamma", "delta"])
result = list(take_until_size(words, 10))
remaining = list(words)

assert result == ["alpha", "beta"]  # 5 + 4 = 9
assert remaining == ["delta"]       # "gamma" was inspected and consumed

result, remaining


(['alpha', 'beta'], ['delta'])

---

# Problem 23 — Buffered budget slicing without losing the first rejected item

## Task

Fix the previous problem by returning the rejected item back to the stream.

Write `take_until_size_buffered(iterable, max_total_chars)` that returns:

```python
accepted_values, replay_iterator
```

where `replay_iterator` begins with the first rejected item, then continues the original iterator.


## Solution 23


In [30]:
def take_until_size_buffered(
    iterable: Iterable[str],
    max_total_chars: int,
) -> tuple[list[str], Iterator[str]]:
    if max_total_chars < 0:
        raise ValueError("max_total_chars must be non-negative")

    iterator = iter(iterable)
    accepted: list[str] = []
    total = 0

    for item in iterator:
        proposed_total = total + len(item)
        if proposed_total > max_total_chars:
            return accepted, chain([item], iterator)
        accepted.append(item)
        total = proposed_total

    return accepted, iter(())


words = iter(["alpha", "beta", "gamma", "delta"])
accepted, replay = take_until_size_buffered(words, 10)

assert accepted == ["alpha", "beta"]
assert list(replay) == ["gamma", "delta"]

accepted


['alpha', 'beta']

---

# Problem 24 — Slice groups, not individual items

## Task

A stream contains batches. You want to select batches `2..4`, then flatten the selected batches into a single list.

## Constraints

- Use `islice` to slice at the batch level.
- Flatten only the selected batches.


## Solution 24


In [31]:
def batch_stream() -> Iterator[list[int]]:
    for group_id in count(0):
        start = group_id * 10
        yield [start, start + 1, start + 2]


selected_batches = islice(batch_stream(), 2, 5)
flattened = [item for batch in selected_batches for item in batch]

assert flattened == [20, 21, 22, 30, 31, 32, 40, 41, 42]
flattened


[20, 21, 22, 30, 31, 32, 40, 41, 42]

---

# Problem 25 — Challenge: parse, filter, and slice a record stream

## Task

You receive a stream of CSV-like text records:

```text
id,score,status
0,80,ok
1,55,retry
2,90,ok
...
```

Build a pipeline that:

1. skips the header,
2. parses rows into dictionaries,
3. keeps only rows with `status == "ok"`,
4. skips the first two ok rows,
5. returns the next five ok rows.

## Constraint

Keep everything lazy until the final `list(...)` call.


## Solution 25


In [32]:
def csv_like_stream(total: int) -> Iterator[str]:
    yield "id,score,status"
    for i in range(total):
        status = "ok" if i % 3 != 1 else "retry"
        score = 50 + (i * 7) % 51
        yield f"{i},{score},{status}"


def parse_record(line: str) -> dict[str, Any]:
    raw_id, raw_score, status = line.split(",")
    return {
        "id": int(raw_id),
        "score": int(raw_score),
        "status": status,
    }


def selected_ok_records(lines: Iterable[str]) -> list[dict[str, Any]]:
    body = islice(lines, 1, None)  # skip header
    parsed = (parse_record(line) for line in body)
    ok_records = (record for record in parsed if record["status"] == "ok")
    return list(islice(ok_records, 2, 7))


records = selected_ok_records(csv_like_stream(30))

assert [record["id"] for record in records] == [3, 5, 6, 8, 9]
assert all(record["status"] == "ok" for record in records)

records


[{'id': 3, 'score': 71, 'status': 'ok'},
 {'id': 5, 'score': 85, 'status': 'ok'},
 {'id': 6, 'score': 92, 'status': 'ok'},
 {'id': 8, 'score': 55, 'status': 'ok'},
 {'id': 9, 'score': 62, 'status': 'ok'}]

---

# Summary: best practices for slicing iterables

- Use normal slicing for sequences such as `list`, `tuple`, `str`, and `range`.
- Use `itertools.islice` for lazy iterables and generators.
- Remember that `islice` returns an iterator, so it is single-use.
- Remember that slicing an iterator consumes the underlying iterator.
- Skipped items are still consumed.
- Use `deque(maxlen=n)` for tail-like behavior on finite iterables.
- Use `tee` or `chain(preview, iterator)` when you need preview/replay behavior.
- Avoid negative slicing assumptions for streams; negative indexes require knowing the end.
- For repeatable slices, use a source factory that creates a fresh iterator.


In [33]:
# Final smoke test: a compact pipeline using several helpers from the notebook.
stream = count(1)
page = get_page((str(n) for n in stream), page_number=3, page_size=4)
windows = list(windowed(page, 2))

assert page == ["13", "14", "15", "16"]
assert windows == [("13", "14"), ("14", "15"), ("15", "16")]

"Notebook smoke test passed"


'Notebook smoke test passed'